<a href="https://www.kaggle.com/code/somnathg25ait2107/mlops-project-gr34?scriptVersionId=326928402" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
!pip install -q transformers datasets wandb huggingface_hub scikit-learn

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch
import wandb
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

In [ ]:
secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
login(token=secrets.get_secret("HF_TOKEN"))
wandb.login()

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
HF_REPO = "somnathchakraborty/distilbert-imdb-sentiment"
VERSION = "final-v1"

EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")

print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"Features: {dataset['train'].features}")

In [ ]:
def clean_text(text):
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


train_df = dataset['train'].to_pandas()
train_df['text'] = train_df['text'].apply(clean_text)
train_df = train_df.sample(n=5000, random_state=42).reset_index(drop=True)
train_df = train_df.drop_duplicates(subset='text').reset_index(drop=True)

test_df = dataset['test'].to_pandas()
test_df['text'] = test_df['text'].apply(clean_text)
test_df = test_df.sample(n=1000, random_state=42).reset_index(drop=True)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"\nClass distribution:\n{train_df['label'].value_counts()}")
print(f"\nMissing values: {train_df.isnull().sum().to_dict()}")

In [ ]:
id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_data(df, max_length=512):
    encodings = tokenizer(
        df['text'].tolist(),
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors='pt',
    )
    return Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': torch.tensor(df['label'].values),
    })


train_dataset = tokenize_data(train_df)
test_dataset = tokenize_data(test_df)

print(f"Train dataset: {len(train_dataset)}")
print(f"Test dataset: {len(test_dataset)}")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
wandb.init(
    entity="ashish-iit-jodhpur-25ait2051",
    project="mlops-assignment3",
    name=f"run-{VERSION}",
    config={
        "model": MODEL_NAME,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_steps": WARMUP_STEPS,
        "version": VERSION,
        "platform": "Kaggle",
    },
)

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name=f"run-{VERSION}",
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1: {results['eval_f1']:.4f}")
print(f"Loss: {results['eval_loss']:.4f}")

In [ ]:
current_accuracy = results['eval_accuracy']

try:
    from huggingface_hub import model_info
    info = model_info(HF_REPO)
    existing_accuracy = float(info.card_data.get("eval_accuracy", 0)) if info.card_data else 0
except Exception:
    existing_accuracy = 0

print(f"Current run accuracy: {current_accuracy:.4f}")
print(f"Existing HF model accuracy: {existing_accuracy:.4f}")

if current_accuracy > existing_accuracy:
    model.push_to_hub(HF_REPO, commit_message=f"run-{VERSION} accuracy={current_accuracy:.4f}")
    tokenizer.push_to_hub(HF_REPO)

    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_file(
        path_or_fileobj=json.dumps({"eval_accuracy": current_accuracy}).encode(),
        path_in_repo="eval_results.json",
        repo_id=HF_REPO,
    )

    hf_url = f"https://huggingface.co/{HF_REPO}"
    wandb.run.summary["huggingface_model"] = hf_url
    print(f"Better model pushed to: {hf_url}")
else:
    print(f"Skipping push — existing model ({existing_accuracy:.4f}) is better or equal")

In [ ]:
wandb.finish()
print("Done!")